# Week 4 - Finite Markov Decision Processes

This notebook introduces a finite Gridworld MDP and compares fixed policies with different behavior quality.

## Context

- Weeks 1-3 covered bandits.
- Bandits only involve action -> reward.
- MDPs introduce state -> action -> reward -> next state.
- This is the transition from bandits to full reinforcement learning.

## Agent-Environment Interface

S<sub>0</sub>, A<sub>0</sub>, R<sub>1</sub>, S<sub>1</sub>, A<sub>1</sub>, R<sub>2</sub>, S<sub>2</sub>, ...

At time step `t`, the agent observes S<sub>t</sub>, takes action A<sub>t</sub>, receives reward R<sub>t+1</sub>, and transitions to S<sub>t+1</sub>. The reward is indexed as R<sub>t+1</sub> because it arrives after the action.

## What Is a Finite MDP?

A finite MDP has finite states and actions, rewards, transitions, and a discount factor. The transition model is `p(s', r | s, a)`.

## Markov Property

The future depends only on the current state and action, not on the full history.

## Rewards and Goals

In this Gridworld, normal steps receive `-1`, reaching the goal receives `+10`, and invalid moves receive `-5`.

## Returns and Discounting

The return is the cumulative future reward. With discounting,

G<sub>t</sub> = R<sub>t+1</sub> + gamma R<sub>t+2</sub> + gamma<sup>2</sup> R<sub>t+3</sub> + ...

## Policies

`pi(a | s)` describes how actions are chosen in each state. In an MDP, policy quality matters because actions affect both immediate rewards and future states.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import matplotlib.pyplot as plt
import numpy as np

from src.gridworld.environment import GridworldMDP
from src.gridworld.policies import BadPolicy, GoalDirectedPolicy, RandomPolicy
from src.gridworld.experiments import (
    run_episode,
    run_policy_evaluation_experiment,
    summarize_policy_results,
)
from src.utils.plotting import (
    plot_average_episode_length_by_policy,
    plot_average_return_by_policy,
    plot_state_visitation_heatmap,
    plot_success_rate_by_policy,
)

In [ ]:
env = GridworldMDP()

print('Grid size:', env.grid_size)
print('Start state:', env.start_state)
print('Goal state:', env.goal_state)
print('Obstacles:', sorted(env.obstacles))
print('Actions:', env.get_valid_actions())
print()
print(env.render())

In [ ]:
random_policy = RandomPolicy(seed=0)
episode = run_episode(env=env, policy=random_policy, max_steps=25, gamma=1.0)

print('States:', episode['states'])
print('Actions:', episode['actions'])
print('Rewards:', episode['rewards'])
print('Return:', episode['return'])
print('Episode length:', episode['episode_length'])
print('Success:', episode['success'])

A single episode is noisy. To compare policies properly, the environment should be sampled many times and summarized with aggregate metrics.

In [ ]:
policy_configs = {
    'Random Policy': {
        'policy_class': RandomPolicy,
        'policy_kwargs': {},
    },
    'Goal-Directed Policy': {
        'policy_class': GoalDirectedPolicy,
        'policy_kwargs': {
            'goal_state': (4, 4),
            'exploration_prob': 0.1,
        },
    },
    'Bad Policy': {
        'policy_class': BadPolicy,
        'policy_kwargs': {},
    },
}

results = run_policy_evaluation_experiment(
    policy_configs=policy_configs,
    n_episodes=500,
    max_steps=100,
    gamma=1.0,
    env_kwargs={},
    seed=0,
)

summary = summarize_policy_results(results)
summary

In [ ]:
for policy_name, metrics in summary.items():
    print(policy_name)
    print(f"  average return: {metrics['average_return']:.2f}")
    print(f"  average episode length: {metrics['average_episode_length']:.2f}")
    print(f"  success rate: {metrics['success_rate'] * 100:.2f}%")
    print()

In [ ]:
results_dir = project_root / 'results' / 'week_04'

plot_average_return_by_policy(
    summary,
    save_path=results_dir / 'average_return_by_policy.png',
)
plot_average_episode_length_by_policy(
    summary,
    save_path=results_dir / 'average_episode_length_by_policy.png',
)
plot_success_rate_by_policy(
    summary,
    save_path=results_dir / 'success_rate_by_policy.png',
)
plot_state_visitation_heatmap(
    results['Goal-Directed Policy']['state_visitation_counts'],
    env,
    title='State Visitation Heatmap - Goal-Directed Policy',
    save_path=results_dir / 'state_visitation_heatmap.png',
)

In [ ]:
plot_average_return_by_policy(summary)
plot_average_episode_length_by_policy(summary)
plot_success_rate_by_policy(summary)
plot_state_visitation_heatmap(
    results['Goal-Directed Policy']['state_visitation_counts'],
    env,
    title='State Visitation Heatmap - Goal-Directed Policy',
)
plt.show()

## Final Interpretation

- The goal-directed policy should achieve the highest return.
- It should also complete episodes faster and succeed more often.
- The random policy acts as a baseline.
- The bad policy performs worse because it tends to move in poor directions.

This shows why policies matter in MDPs: actions change both immediate rewards and the future states the agent will face.

## Technical Takeaways

- MDPs introduce states and transitions.
- Return measures cumulative future reward.
- `gamma` controls how much future rewards matter.
- Policies map states to actions.
- Different policies can produce very different long-term outcomes in the same environment.